# NovaByte Retail S.L. - Auditoría y Análisis Exploratorio de Inventarios
### Práctica RA1 - M3 Programació d'Intel·ligència Artificial

---

## 📌 Objetivo de este Notebook
En esta fase exploratoria **cargamos la totalidad de los datos** procedentes de los cuatro centros logísticos (Barcelona, Madrid, Valencia y Sevilla) en formatos **CSV** y **JSON**, **sin eliminar ni descartar ningún registro**.

El propósito es realizar una auditoría integral que cubra:
1. **Análisis general de todos los productos**: distribución por centros, categorías, estados y métricas descriptivas globales.
2. **Detección exhaustiva de anomalías y productos sospechosos**: valores astronómicamente altos (como `stock_actual = 99999`), precios unitarios desproporcionados, valores negativos, nulos y duplicados.
3. **Auditoría de calidad por centro**: comparación de la fiabilidad de datos entre sedes para diseñar las reglas de negocio de la posterior automatización.

> **Compatibilidad Google Colab**: Este notebook está preparado para ejecutarse tanto localmente en Jupyter / VS Code como en **Google Colab**.

In [ ]:
# =============================================================================
# 0. Configuración del entorno y librerías
# =============================================================================
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de visualización
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# Si estás ejecutando en Google Colab y necesitas subir los archivos, descomenta:
# from google.colab import files
# print("Sube los archivos: inventario_BCN.csv, inventario_MAD.csv, inventario_VAL.json, inventario_SEV.json")
# uploaded = files.upload()

print("Entorno configurado correctamente.")

---
## 📥 1. Carga de Datasets en Bruto (Sin Borrado de Datos)

Cargamos los datos desde los dos formatos estipulados (**CSV** y **JSON**) manteniendo cada registro exactamente como se encuentra en el archivo origen, añadiendo únicamente metadatos sobre su origen para poder auditar cada centro.

In [ ]:
# Funciones de lectura modulares para CSV y JSON
def cargar_csv_raw(ruta: str, centro: str) -> pd.DataFrame:
    """Carga un dataset CSV sin filtrar ni descartar registros."""
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"No se encontró el archivo: {ruta}")
    df = pd.read_csv(ruta, encoding="utf-8")
    df["centro_origen"] = centro
    df["formato_origen"] = "CSV"
    df["archivo_origen"] = os.path.basename(ruta)
    return df

def cargar_json_raw(ruta: str, centro: str) -> pd.DataFrame:
    """Carga un dataset JSON sin filtrar ni descartar registros."""
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"No se encontró el archivo: {ruta}")
    with open(ruta, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    df["centro_origen"] = centro
    df["formato_origen"] = "JSON"
    df["archivo_origen"] = os.path.basename(ruta)
    return df

# Carga de los 4 centros de NovaByte Retail
df_bcn = cargar_csv_raw("inventario_BCN.csv", "Barcelona")
df_mad = cargar_csv_raw("inventario_MAD.csv", "Madrid")
df_val = cargar_json_raw("inventario_VAL.json", "Valencia")
df_sev = cargar_json_raw("inventario_SEV.json", "Sevilla")

# Unificación en un DataFrame Maestro (3000 registros intactos)
df_inventario = pd.concat([df_bcn, df_mad, df_val, df_sev], ignore_index=True)

print("Resumen de carga:")
print(f" - Barcelona (CSV): {len(df_bcn)} registros")
print(f" - Madrid    (CSV): {len(df_mad)} registros")
print(f" - Valencia  (JSON): {len(df_val)} registros")
print(f" - Sevilla   (JSON): {len(df_sev)} registros")
print(f"TOTAL EN BRUTO: {len(df_inventario)} productos cargados sin pérdida de información.")
df_inventario.head()

---
## 📊 2. Análisis General de Todos los Productos

En esta sección realizamos un análisis descriptivo del catálogo completo, evaluando la distribución de los productos por categoría, estado comercial, almacén y proveedor.

In [ ]:
# Información estructural y tipos de datos
print("=== INFORMACIÓN ESTRUCTURAL DEL CATÁLOGO ===")
df_inventario.info()

# Conversión no destructiva a tipos numéricos para análisis estadístico
df_inventario["precio_num"] = pd.to_numeric(df_inventario["precio_unitario"], errors="coerce")
df_inventario["stock_actual_num"] = pd.to_numeric(df_inventario["stock_actual"], errors="coerce")
df_inventario["stock_minimo_num"] = pd.to_numeric(df_inventario["stock_minimo"], errors="coerce")

print("\n=== RESUMEN ESTADÍSTICO GENERAL (EN BRUTO) ===")
display(df_inventario[["precio_num", "stock_actual_num", "stock_minimo_num"]].describe())

In [ ]:
# Distribución de productos por Categoría y por Centro
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Productos por Almacén
sns.countplot(data=df_inventario, x="centro_origen", hue="formato_origen", ax=axes[0], palette="Set2")
axes[0].set_title("Volumen de Productos por Centro Logístico y Formato", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Almacén")
axes[0].set_ylabel("Cantidad de Productos")

# Gráfico 2: Categorías (frecuencia)
cat_counts = df_inventario["categoria"].value_counts(dropna=False)
sns.barplot(x=cat_counts.values, y=cat_counts.index.astype(str), ax=axes[1], palette="viridis")
axes[1].set_title("Distribución de Productos por Categoría (en Bruto)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Número de Registros")
axes[1].set_ylabel("Categoría")

plt.tight_layout()
plt.show()

print("Conteo detallado de categorías detectadas (incluye problemas de mayúsculas/minúsculas y valores no estándar):")
print(df_inventario["categoria"].value_counts(dropna=False))

In [ ]:
# Distribución por Estado Comercial
plt.figure(figsize=(10, 4))
estado_counts = df_inventario["estado"].value_counts(dropna=False)
sns.barplot(x=estado_counts.index.astype(str), y=estado_counts.values, palette="magma")
plt.title("Distribución de Estados Comerciales en Bruto", fontsize=13, fontweight="bold")
plt.xlabel("Estado")
plt.ylabel("Registros")
for i, val in enumerate(estado_counts.values):
    plt.text(i, val + 20, str(val), ha="center", fontweight="bold")
plt.show()

print("Estados encontrados:")
print(estado_counts)

---
## 🚨 3. Detección y Análisis de Datos Sospechosamente Demasiado Altos (Outliers Extremos)

Una de las observaciones más críticas al analizar el inventario en bruto es la presencia de **valores anormalmente gigantescos o desproporcionados**.

En particular:
* **Stock astronómico de 99.999 unidades**: En un almacén de electrónica retail, un valor de 99.999 unidades es claramente un dato centinela (*placeholder*) o un error de tecleo.
* **Distorsión del valor total**: Este stock ficticio infla el valor del inventario en cientos de millones de euros.

In [ ]:
# Detección de productos con stock sospechosamente excesivo (stock > 500)
outliers_stock = df_inventario[df_inventario["stock_actual_num"] > 500].copy()

print(f"⚠️ Se han encontrado {len(outliers_stock)} productos con stock sospechosamente desproporcionado:")
cols_mostrar = ["centro_origen", "producto_id", "nombre", "categoria", "precio_unitario", "stock_actual", "stock_minimo"]
display(outliers_stock[cols_mostrar])

# Comparación del impacto en la valoración total del inventario
df_inventario["valor_ficticio"] = df_inventario["precio_num"] * df_inventario["stock_actual_num"]

valor_total_con_outliers = df_inventario["valor_ficticio"].sum()
valor_sin_outliers = df_inventario.loc[df_inventario["stock_actual_num"] <= 500, "valor_ficticio"].sum()
impacto_distorsion = valor_total_con_outliers - valor_sin_outliers

print(f"\n💰 IMPACTO ECONÓMICO DE LA ANOMALÍA:")
print(f" - Valor total del inventario (CON stock 99999):   {valor_total_con_outliers:,.2f} €")
print(f" - Valor real aproximado (SIN stock 99999):        {valor_sin_outliers:,.2f} €")
print(f" - Distorsión introducida por estos 6 registros:  {impacto_distorsion:,.2f} € (¡Más de 400 millones de euros ficticios!)")

In [ ]:
# Visualización de la distorsión del stock (Con vs Sin Outliers Extremos)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Con outliers extremos (99999)
sns.boxplot(x="centro_origen", y="stock_actual_num", data=df_inventario, ax=axes[0], palette="Reds")
axes[0].set_title("Stock Actual CON Outliers Extremos (99.999)", fontsize=12, fontweight="bold", color="darkred")
axes[0].set_ylabel("Stock Actual (Unidades)")

# Sin outliers extremos para ver la distribución real de almacén
df_stock_normal = df_inventario[df_inventario["stock_actual_num"].between(0, 500)]
sns.boxplot(x="centro_origen", y="stock_actual_num", data=df_stock_normal, ax=axes[1], palette="Blues")
axes[1].set_title("Stock Actual Distribución Normal (0 - 200 unidades)", fontsize=12, fontweight="bold", color="navy")
axes[1].set_ylabel("Stock Actual (Unidades)")

plt.tight_layout()
plt.show()

In [ ]:
# Análisis de Precios Unitarios Máximos y Sospechosos
top_precios = df_inventario.dropna(subset=["precio_num"]).sort_values(by="precio_num", ascending=False).head(15)

print("Top 15 productos con el precio unitario más alto:")
display(top_precios[["centro_origen", "producto_id", "nombre", "categoria", "precio_num", "stock_actual"]])

# Gráfico de dispersión: Precio Unitario vs Stock Actual
plt.figure(figsize=(12, 6))
sns.scatterplot(
    data=df_inventario,
    x="precio_num",
    y="stock_actual_num",
    hue="centro_origen",
    alpha=0.7,
    s=60
)
plt.title("Dispersión Global: Precio Unitario vs Stock Actual", fontsize=13, fontweight="bold")
plt.xlabel("Precio Unitario (€)")
plt.ylabel("Stock Actual (Unidades)")
plt.axhline(500, color="red", linestyle="--", label="Umbral de sospecha stock (>500)")
plt.legend()
plt.show()

---
## 🔍 4. Detección de Otras Anomalías Críticas

Además de los valores gigantescos, la auditoría requiere detectar:
1. **Valores negativos e imposibles** (`precio < 0`, `stock_actual < 0`, `stock_minimo < 0`).
2. **Valores nulos, vacíos o campos ausentes** (`NaN`, cadenas vacías, fechas omitidas).
3. **Identificadores duplicados** (`producto_id` duplicado).
4. **Inconsistencias en nombres de categoría y estados**.

In [ ]:
# 4.1. Valores Negativos e Imposibles
precios_negativos = df_inventario[df_inventario["precio_num"] < 0]
stock_negativo = df_inventario[df_inventario["stock_actual_num"] < 0]
stock_min_negativo = df_inventario[df_inventario["stock_minimo_num"] < 0]

print(f"⚠️ Precios unitarios negativos: {len(precios_negativos)} casos")
display(precios_negativos[["centro_origen", "producto_id", "nombre", "precio_unitario"]])

print(f"\n⚠️ Stock actual negativo: {len(stock_negativo)} casos")
display(stock_negativo[["centro_origen", "producto_id", "nombre", "stock_actual"]])

print(f"\n⚠️ Stock mínimo negativo: {len(stock_min_negativo)} casos")
display(stock_min_negativo[["centro_origen", "producto_id", "nombre", "stock_minimo"]])

In [ ]:
# 4.2. Valores Nulos y Campos Ausentes por Almacén
nulos_por_centro = df_inventario.groupby("centro_origen")[["producto_id", "nombre", "categoria", "precio_unitario", "stock_actual", "stock_minimo", "fecha_ultima_entrada"]].apply(lambda g: g.isna().sum() + (g == "").sum())

print("=== CONTEO DE VALORES NULOS O VACÍOS POR CENTRO LOGÍSTICO ===")
display(nulos_por_centro)

# Heatmap de datos ausentes
plt.figure(figsize=(10, 4))
sns.heatmap(nulos_por_centro.T, annot=True, fmt="d", cmap="YlOrRd", cbar=True)
plt.title("Mapa de Incidencias: Datos Nulos / Ausentes por Centro", fontsize=13, fontweight="bold")
plt.xlabel("Centro Logístico")
plt.ylabel("Campo del Registro")
plt.show()

In [ ]:
# 4.3. Identificadores Duplicados
# Consideramos IDs válidos (no nulos ni vacíos) que se repiten
ids_validos = df_inventario[df_inventario["producto_id"].notna() & (df_inventario["producto_id"] != "")]
duplicados_id = ids_validos[ids_validos.duplicated(subset=["producto_id"], keep=False)].sort_values(by="producto_id")

print(f"⚠️ Se detectaron {len(duplicados_id)} registros con 'producto_id' DUPLICADO:")
display(duplicados_id[["centro_origen", "producto_id", "nombre", "categoria", "precio_unitario", "stock_actual"]])

In [ ]:
# 4.4. Inconsistencias de Categorías y Estados (Calidad de Etiquetas)
categorias_validas_oficiales = {"Gaming", "Informática", "Telefonía", "Accesorios", "Audio", "Hogar inteligente"}
estados_validos_oficiales = {"activo", "descatalogado"}

# Detección de anomalías en categorías
cats_anomalas = df_inventario[~df_inventario["categoria"].isin(categorias_validas_oficiales)]
print(f"⚠️ Categorías no normalizadas o anómalas ({len(cats_anomalas)} registros):")
print(cats_anomalas["categoria"].value_counts(dropna=False))

# Detección de anomalías en estados
estados_anomalos = df_inventario[~df_inventario["estado"].isin(estados_validos_oficiales)]
print(f"\n⚠️ Estados comerciales no estandarizados ({len(estados_anomalos)} registros):")
print(estados_anomalos["estado"].value_counts(dropna=False))

---
## 📋 5. Cuadro de Calidad de Datos (Data Quality Scorecard) por Almacén

Clasificamos cada registro del inventario según su nivel de calidad:
* **Limpio / Correcto**: Pasa todas las validaciones sin incidencias.
* **Incidencia Leve**: Recuperable (ej. mayúsculas en categoría o estado 'ACTIVO').
* **Incidencia Crítica**: Stock negativo, stock 99999, precio negativo, campo obligatorio vacío o ID duplicado.

In [ ]:
def evaluar_calidad_registro(row, ids_duplicados_set) -> str:
    pid = str(row["producto_id"]).strip() if pd.notna(row["producto_id"]) else ""
    precio = row["precio_num"]
    stock = row["stock_actual_num"]
    stock_min = row["stock_minimo_num"]
    
    # Errores críticos
    if not pid or pid in ids_duplicados_set:
        return "Crítico: Error en ID (nulo o duplicado)"
    if pd.isna(precio) or precio < 0:
        return "Crítico: Precio inválido (nulo o negativo)"
    if pd.isna(stock) or stock < 0 or stock > 500:
        return "Crítico: Stock inválido (nulo, negativo o 99999)"
    if pd.isna(stock_min) or stock_min < 0:
        return "Crítico: Stock mínimo inválido"
    if pd.isna(row["nombre"]) or str(row["nombre"]).strip() == "":
        return "Crítico: Nombre ausente"
    
    # Incidencias leves / formato
    cat = str(row["categoria"]).strip()
    est = str(row["estado"]).strip()
    if cat.upper() in ["ACCESORIOS", "AUDIO", "GAMING", "HOGAR INTELIGENTE", "INFORMÁTICA", "TELEFONÍA"] and cat not in categorias_validas_oficiales:
        return "Leve: Categoría en mayúsculas (normalizable)"
    if est.upper() == "ACTIVO" and est != "activo":
        return "Leve: Estado en mayúsculas (normalizable)"
    if cat in ["Otros??", "MISC", "Electrodomésticos"] or est in ["X", "retirado?", "pendiente"]:
        return "Leve: Categoría/Estado atípico"
    
    return "Correcto: Válido"

dups_set = set(duplicados_id["producto_id"].unique())
df_inventario["clasificacion_calidad"] = df_inventario.apply(evaluar_calidad_registro, ids_duplicados_set=dups_set, axis=1)

# Resumen porcentual por centro
resumen_calidad = pd.crosstab(
    df_inventario["centro_origen"],
    df_inventario["clasificacion_calidad"].apply(lambda x: "Crítico" if "Crítico" in x else ("Leve" if "Leve" in x else "Correcto")),
    margins=True,
    margins_name="TOTAL"
)

print("=== CUADRO DE CALIDAD DE DATOS POR CENTRO ===")
display(resumen_calidad)

# Gráfico comparativo
calidad_pct = pd.crosstab(
    df_inventario["centro_origen"],
    df_inventario["clasificacion_calidad"].apply(lambda x: "Crítico" if "Crítico" in x else ("Leve" if "Leve" in x else "Correcto")),
    normalize="index"
) * 100

calidad_pct.plot(kind="bar", stacked=True, color=["#2ca02c", "#d62728", "#ff7f0e"], figsize=(12, 6))
plt.title("Porcentaje de Calidad de Datos por Centro Logístico", fontsize=13, fontweight="bold")
plt.xlabel("Almacén")
plt.ylabel("Porcentaje (%)")
plt.legend(title="Calidad", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

---
## 💡 6. Conclusiones y Reglas para la Automatización (`funciones.py`)

A partir de este análisis exploratorio, extraemos las siguientes conclusiones clave para justificar en el `README.md` y programar en la fase final:

1. **Centros de referencia limpios**:
   * **Barcelona (CSV)** y **Valencia (JSON)** tienen un 100% de datos válidos. Sirven como prueba para verificar el flujo de procesamiento normal.

2. **Centros con incidencias intencionadas**:
   * **Madrid (CSV)**: Contiene valores nulos de precio, precios negativos, stocks negativos, duplicados de `producto_id` y valores centinela `99999`.
   * **Sevilla (JSON)**: Presenta campos ausentes (`fecha_ultima_entrada`), valores nulos, stocks centinela `99999`, stocks negativos y categorías/estados en mayúsculas.

3. **Decisiones de negocio recomendadas**:
   * **Stock astronómico (`99999`)**: Marcar como inválido o solicitar revisión manual, ya que distorsiona en cientos de millones el valor del almacén.
   * **Precios/Stock negativos**: Descartar el registro para cálculos financieros o marcar como inválido, emitiendo un aviso.
   * **Categorías y estados en mayúsculas**: Normalizar automáticamente mediante `.capitalize()` o `.lower()` para unificar el catálogo sin perder datos válidos.
   * **Campos opcionales ausentes** (ej. fecha de entrada): Asignar un valor por defecto (ej. `"Sin fecha"` o `None`) sin invalidar el producto si el resto de datos clave son válidos.